## 图书域交互数据预处理（2023版）
流程：加载 → 仅保留有meta的物品 → 迭代式双向质量过滤 → 保存中间结果

In [1]:
import json, gzip, csv, os
import sys
from collections import Counter
import pandas as pd

REVIEW_PATH = "./Books.jsonl.gz"
META_PATH = "./processed/book_meta_clean.csv"
SAVE_DIR = "./processed/"
os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
# 加载干净meta
# 在 Windows 上 sys.maxsize 可能太大，会导致 csv.field_size_limit 报 OverflowError
import sys
import csv
max_int = sys.maxsize
while True:
    try:
        csv.field_size_limit(max_int)
        break
    except OverflowError:
        max_int = max_int // 10

with open(META_PATH, encoding="utf-8") as f:
    meta_items = set()
    for row in csv.DictReader(f):
        meta_items.add(row["parent_asin"])
print(f"有meta的图书物品: {len(meta_items)}")


有meta的图书物品: 3082232


In [3]:
# 遍历交互数据，仅做meta过滤
rows = []
total = 0
has_meta = 0

with gzip.open(REVIEW_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        total += 1
        try:
            obj = json.loads(line.strip())
        except:
            continue
        
        parent_asin = obj.get("parent_asin", "")
        if parent_asin not in meta_items:
            continue
        has_meta += 1
        
        rows.append({
            "user_id": obj["user_id"],
            "item_id": parent_asin,
            "timestamp": int(float(obj.get("timestamp", 0)) / 1000.0),
            "domain": "book"
        })
        
        if total % 5000000 == 0:
            print(f"  已处理 {total/1e6:.1f}M, 有meta {has_meta} ...")

print(f"总交互: {total}")
print(f"有meta: {has_meta} ({has_meta/total*100:.1f}%)")

df = pd.DataFrame(rows)
del rows
print(f"当前: {len(df)} 条交互, {df['user_id'].nunique()} 用户, {df['item_id'].nunique()} 物品")

  已处理 5.0M, 有meta 4424451 ...
  已处理 10.0M, 有meta 8848672 ...
  已处理 15.0M, 有meta 13264802 ...
  已处理 25.0M, 有meta 21614635 ...
总交互: 29475453
有meta: 25093929 (85.1%)
当前: 25093929 条交互, 9225991 用户, 3080439 物品


In [4]:
# ===== 迭代式双向过滤 =====
MIN_ITEM = 5
MIN_USER = 5

while True:
    n_u_before = df["user_id"].nunique()
    n_i_before = df["item_id"].nunique()
    
    item_cnt = df.groupby("item_id").size()
    df = df[df["item_id"].isin(item_cnt[item_cnt >= MIN_ITEM].index)]
    
    user_cnt = df.groupby("user_id").size()
    df = df[df["user_id"].isin(user_cnt[user_cnt >= MIN_USER].index)]
    
    if df["user_id"].nunique() == n_u_before and df["item_id"].nunique() == n_i_before:
        break
    print(f"  迭代中... 用户={df['user_id'].nunique()}, 物品={df['item_id'].nunique()}")

print(f"\n===== 图书域过滤完成 =====")
print(f"交互: {len(df)}")
print(f"用户: {df['user_id'].nunique()}")
print(f"物品: {df['item_id'].nunique()}")

  迭代中... 用户=819171, 物品=844758
  迭代中... 用户=697922, 物品=466681
  迭代中... 用户=681930, 物品=435216
  迭代中... 用户=679134, 物品=430196
  迭代中... 用户=678596, 物品=429264
  迭代中... 用户=678484, 物品=429055
  迭代中... 用户=678460, 物品=429010
  迭代中... 用户=678456, 物品=429003
  迭代中... 用户=678456, 物品=429001

===== 图书域过滤完成 =====
交互: 8145379
用户: 678456
物品: 429001


In [5]:
# 保存图书域中间结果
df.to_csv(f'{SAVE_DIR}/book_interactions_filtered.csv', index=False)
print(f'已保存: {SAVE_DIR}/book_interactions_filtered.csv')

已保存: ./processed//book_interactions_filtered.csv
